**Aceleradores de Hardware - Ordem de Custo**

CPU

GPUs: T4

GPUs: L4

TPU v5e-1

TPU v6e-1

GPUs: A100

GPUs: G4

GPUs: H100

### Importar bibliotecas e definir variaveis iniciais

In [ ]:
#!pip install transformers

In [ ]:
# Importa bibliotecas necessarias
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pylab import rcParams

from sklearn.metrics import confusion_matrix, classification_report

# Bibliotecas de ML com PyTorch
from transformers import BertModel, AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader

# Diversos
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Define variaveis iniciais e constantes
%config InlineBackend.figure_format='retina'

# Configuracao dos graficos
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
HAPPY_COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(HAPPY_COLORS_PALETTE))
rcParams['figure.figsize'] = 12, 8

# Semente aleatoria para reprodutibilidade
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Define dispositivo (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


##Parametrização conforme ambiente de execução.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_PATH_GENERAL = '/content/drive/MyDrive/_Mestrado PUCPR/'
BASE_PATH_DATASET = f'{BASE_PATH_GENERAL}Datasets/'

LISTA_MODEL_CHECKPOINT = [
    {
        'endpoint': 'raquelsilveira/legalbertpt_fp',
        'description': 'LegalBERTpt Jurídico',
        'layers_to_freeze': 12,
        'fine_tunning': True,
        'head_tail_approach': None
    },
    {
        'endpoint': 'raquelsilveira/legalbertpt_fp',
        'description': 'LegalBERTpt Jurídico (Head Tail Approach)',
        'layers_to_freeze': 12,
        'fine_tunning': True,
        'head_tail_approach': 512
    },
    {
        'endpoint': 'rufimelo/Legal-BERTimbau-base',
        'description': 'Legal-BERTimbau Jurídico Base',
        'layers_to_freeze': 12,
        'fine_tunning': False,
        'head_tail_approach': None
    },
    {
        'endpoint': 'rufimelo/Legal-BERTimbau-large',
        'description': 'Legal-BERTimbau Jurídico Large',
        'layers_to_freeze': 24,
        'fine_tunning': False,
        'head_tail_approach': None
    },
    {
        'endpoint': 'neuralmind/bert-base-portuguese-cased',
        'description': 'BERTimbau Base',
        'layers_to_freeze': 12,
        'fine_tunning': False,
        'head_tail_approach': None
    },
    {
        'endpoint': 'neuralmind/bert-large-portuguese-cased',
        'description': 'BERTimbau Large',
        'layers_to_freeze': 24,
        'fine_tunning': False,
        'head_tail_approach': None
    },
    {
        'endpoint': 'google-bert/bert-base-multilingual-cased',
        'description': 'BERT Multilingual Base',
        'layers_to_freeze': 12,
        'fine_tunning': False,
        'head_tail_approach': None
    },
    {
        'endpoint': 'severinsimmler/xlm-roberta-longformer-base-16384',
        'description': 'XLM-R Longformer Multilingual Base',
        'layers_to_freeze': 12,
        'fine_tunning': False,
        'head_tail_approach': None
    },
]

RANDOM_STATE = 42
EPOCHS = 2
APPROACH_EXECUTION = 'epoch' #'step'
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING = 5
class_names = ['Desfavorável ao Réu', 'Favorável ao Réu']
MAX_LEN = 512

# Performance knobs
NUM_WORKERS = 2

In [ ]:
import os

# Paths padrão para reprodutibilidade
train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2026_train-punct.csv"
test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2026_test-punct.csv"

# Apenas carrega os arquivos já persistidos (sem split)
if not os.path.exists(train_path):
    raise FileNotFoundError(f"Arquivo de treino não encontrado: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Arquivo de teste não encontrado: {test_path}")

print("Carregando arquivos de treino e teste persistidos...")
df_train = pd.read_csv(train_path, sep=';')
df_test = pd.read_csv(test_path, sep=';')

print(f"Treino carregado de: {train_path} | shape: {df_train.shape}")
print(f"Teste carregado de: {test_path} | shape: {df_test.shape}")

Arquivos de treino e teste encontrados. Reutilizando arquivos persistidos...
Treino carregado de: /content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_acordaos_HC_under_2021_2026_treino.csv | shape: (4976, 22)
Teste carregado de: /content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_acordaos_HC_under_2021_2026_teste.csv | shape: (1244, 22)
Dataset combinado (df) | shape: (6220, 22)


In [ ]:
# Funcao para converter a decisao em valor inteiro (0 = Desfavoravel, 1 = Favoravel)
def cast_decisao_to_int(decisao_favoravel):
    if decisao_favoravel == 'S':
        return 1
    else:
        return 0

# Aplica a transformacao em treino e teste
if 'DECISAO_FAVORAVEL' in df_train.columns and 'classe' not in df_train.columns:
    df_train['classe'] = df_train.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)
if 'DECISAO_FAVORAVEL' in df_test.columns and 'classe' not in df_test.columns:
    df_test['classe'] = df_test.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)

print(f"Classes no treino: {df_train['classe'].value_counts().to_dict()}")
print(f"Classes no teste:  {df_test['classe'].value_counts().to_dict()}")

### Preparando o Dataset para PyTorch

Para inserir dados no PyTorch, precisamos de uma classe geradora mais robusta. Tambem retornamos o texto do acordao para facilitar a validacao das predicoes.

In [ ]:
from typing import Dict, Union
import numpy as np
import torch
from torch.utils.data import Dataset

class HCAcordaoSTJ(Dataset):
    def __init__(self, reports: np.ndarray, targets: np.ndarray, tokenizer, max_len: int) -> Dataset:
        self.reports = reports.astype(str)
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.reports)

    def __getitem__(self, item: int) -> Dict[str, Union[str, torch.Tensor]]:
        encoding = self.tokenizer(
            self.reports[item],
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True
        )

        return {
            'report_text': self.reports[item],
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'targets': int(self.targets[item])
        }

Criar 80% de dados para treino e 20% para teste.

In [ ]:
# Split já é tratado na célula de carga inteligente (Célula 10).
# Aqui apenas conferimos os dataframes finais para manter compatibilidade do fluxo.
if 'df_train' not in globals() or 'df_test' not in globals():
    raise RuntimeError("Execute primeiro a célula de carga inteligente para criar/carregar treino e teste.")

print(f"Shape de treino: {df_train.shape}")
print(f"Shape de teste: {df_test.shape}")

Train shape: (4976, 22)
Test shape: (1244, 22)


In [ ]:
#Você parou nesta etapa. Agora vc precisa estudar em detalhes a classe DataLoader e as implicações com o treinamento.

Criar um dataloader para liberar os dados em lotes.

### O papel do DataLoader neste treinamento

O `DataLoader` em PyTorch é uma utilidade fundamental para o treinamento de modelos de aprendizado profundo, especialmente com grandes conjuntos de dados. No contexto deste treinamento BERT, ele desempenha vários papéis cruciais:

1.  **Iteração em Lotes (Batching)**: Ele agrupa os dados em pequenos lotes (definidos por `batch_size`). Em vez de alimentar uma única amostra para o modelo por vez (o que é ineficiente para GPUs), o `DataLoader` fornece um lote de amostras. Isso permite que o modelo processe várias entradas simultaneamente, acelerando o treinamento e aproveitando a paralelização do hardware.

2.  **Embaralhamento (Shuffling)**: Antes de cada época de treinamento, o `DataLoader` (se configurado com `shuffle=True`, embora neste notebook esteja implícito que não está explícito no `create_data_loader` mas é uma prática comum para o treinamento) embaralha os dados. Isso é vital para garantir que o modelo não aprenda a ordem dos dados e para evitar que a ordem das amostras introduza vieses no treinamento.

3.  **Processamento Multiprocessado (`num_workers`)**: O parâmetro `num_workers` permite que os dados sejam carregados e pré-processados em paralelo por múltiplos processos secundários (em vez do processo principal). Isso é particularmente útil quando o pré-processamento de dados é uma etapa demorada (como a tokenização e o padding feitos na classe `HCAcordaoSTJ`), pois evita gargalos no carregamento de dados que poderiam deixar a GPU ociosa.

4.  **Integração com `Dataset`**: O `DataLoader` trabalha em conjunto com a classe `Dataset` (neste caso, `HCAcordaoSTJ`). A classe `Dataset` é responsável por como as amostras individuais são carregadas e pré-processadas (tokenização, truncamento, padding, etc.), enquanto o `DataLoader` é responsável por como essas amostras individuais são agrupadas e fornecidas ao modelo em lotes.

Em resumo, o `DataLoader` otimiza o fluxo de dados do disco para a memória e, em seguida, para o modelo, garantindo que o treinamento seja eficiente, reprodutível (com `random_state`) e que o modelo veja os dados de forma variada ao longo das épocas, o que é fundamental para um bom desempenho e generalização.

In [ ]:
import os

def build_dynamic_padding_collate_fn(tokenizer):
    padder = DataCollatorWithPadding(tokenizer=tokenizer, padding='longest', return_tensors='pt')

    def collate_fn(features):
        token_features = [
            {
                'input_ids': f['input_ids'],
                'attention_mask': f['attention_mask']
            }
            for f in features
        ]
        batch = padder(token_features)
        batch['targets'] = torch.tensor([f['targets'] for f in features], dtype=torch.long)
        batch['report_text'] = [f['report_text'] for f in features]
        return batch

    return collate_fn


def create_data_loader(
    df: pd.DataFrame,
    tokenizer,
    max_len: int,
    batch_size: int,
    shuffle: bool = False,
    text_column: str = 'RELATORIO'
) -> DataLoader:
    if text_column not in df.columns:
        raise KeyError(f"Coluna de texto nao encontrada no dataframe: {text_column}")

    ds = HCAcordaoSTJ(
        reports=df[text_column].to_numpy(),
        targets=df.classe.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )

    workers = min(NUM_WORKERS, os.cpu_count() or 1)
    collate_fn = build_dynamic_padding_collate_fn(tokenizer)

    loader_kwargs = {
        'dataset': ds,
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': workers,
        'pin_memory': (device.type == 'cuda'),
        'collate_fn': collate_fn
    }

    if workers > 0:
        loader_kwargs['persistent_workers'] = True
        loader_kwargs['prefetch_factor'] = 2

    return DataLoader(**loader_kwargs)

## Classificacao de Decisao com Transformers (BERT e afins)

Usaremos backbones Hugging Face via `AutoModel` e construiremos um classificador para prever se a decisão será favorável ou desfavorável ao réu.

##### Carregar modelo base (referencia)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

Usamos uma camada de dropout para regularizacao e uma camada totalmente conectada para a saida. Retornamos a saida bruta da ultima camada, pois isso e necessario para a funcao de perda de entropia cruzada no PyTorch. Em seguida, crie uma instancia e mova para a GPU.

In [ ]:
# Bloco de testes alternativos de modelo (mantido apenas como referência)
# from transformers import AutoModelForSequenceClassification
# bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

In [ ]:
# num_layers_to_freeze = LAYERS_TO_FREEZE
# print(f"Freezing the embeddings and the first {num_layers_to_freeze} encoder layers of the model...")

# for name, param in bert_model.named_parameters():
#     if 'embeddings' in name:
#         param.requires_grad = False
#     elif 'encoder.layer' in name:
#         # Extrai o número da camada pelo nome (ex: bert.encoder.layer.0.attention...)
#         layer_num = int(name.split('.')[3])
#         if layer_num < num_layers_to_freeze:
#             param.requires_grad = False
#         else:
#             # Garante que camadas subsequentes sejam treináveis
#             param.requires_grad = True
#     else:
#         # O pooler e o classifier (camadas finais) continuam treináveis
#         param.requires_grad = True

# print("Layers freezing complete. You can now proceed with training.")

In [ ]:
# Imprimir nomes das camadas
# O indice ajuda a identificar quais camadas correspondem aos blocos iniciais (ex.: embedding e primeiras camadas do encoder)
# i = 0
# for name, param in bert_model.named_parameters():
#   i = i + 1
#   print(f"index: {i} {name} - status: {param.requires_grad}")

In [ ]:
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

### O papel da classe `HCAcordaoClassifier` no Fine-Tuning

A classe `HCAcordaoClassifier` e o coracao do seu processo de *fine-tuning*. Ela herda de `nn.Module` (a classe base para todas as redes neurais no PyTorch) e tem a funcao de **adaptar um modelo de linguagem para a sua tarefa especifica** de classificacao binaria (Favoravel vs. Desfavoravel).

Com as modificacoes recentes, a arquitetura passou a ser **dinamica por modelo**. Aqui esta o que cada parte faz:

1. **Base pre-treinada (`self.bert`)**
   * O backbone e carregado com `AutoModel.from_pretrained()`.
   * Isso permite usar BERT, XLM-R Longformer e outros modelos compativeis no mesmo fluxo.

2. **Numero de camadas do modelo (`self.num_layers_model`)**
   * O total de camadas nao vem mais da lista manualmente.
   * Ele e inferido de `self.bert.config.num_hidden_layers`.

3. **Cabeca classificadora (MLP)**
   * A cabeca segue o bloco `Linear -> ReLU -> Dropout` em 3 etapas (`fc1`, `fc2`, `fc3`) e camada final `out`.
   * O tamanho da cabeca e escolhido com base em `self.num_layers_model` (perfil 12 ou 24, com fallback para o perfil mais proximo).

4. **Controle explicito de congelamento (`layers_to_freeze`)**
   * O campo `layers_to_freeze` em `LISTA_MODEL_CHECKPOINT` define quantas camadas iniciais serao congeladas.
   * O congelamento efetivo usa `min(layers_to_freeze, self.num_layers_model)` para evitar inconsistencias.

5. **Fluxo no `forward`**
   * Recebe `input_ids` e `attention_mask`.
   * Usa `pooler_output` quando disponivel; se nao existir, usa fallback para `last_hidden_state[:, 0, :]`.
   * Em seguida aplica a cabeca MLP e retorna os logits finais.

Em resumo, o `HCAcordaoClassifier` e o "tradutor" que pega os dados brutos do mundo real e os transforma em um modelo de classificacao eficiente e robusto.

In [ ]:
class HCAcordaoClassifier(nn.Module):

    # Mapa de arquitetura da cabeça classificadora por número de camadas do modelo
    HEAD_CONFIG = {
        12: (2000, 1000, 500, 250),
        24: (4000, 2000, 1000, 500),
    }

    def __init__(self, n_classes: int, layers_to_freeze: int, model_name: str) -> None:
        super(HCAcordaoClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.num_layers_model = int(getattr(self.bert.config, 'num_hidden_layers', 12))

        # Fallback para arquiteturas com número de camadas diferente de 12 ou 24
        closest_key = min(self.HEAD_CONFIG.keys(), key=lambda k: abs(k - self.num_layers_model))
        h1, h2, h3, h4 = self.HEAD_CONFIG[closest_key]


        self.fc1 = nn.Linear(self.bert.config.hidden_size, h1)
        self.drop1 = nn.Dropout(p=0.3)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(h1, h2)
        self.drop2 = nn.Dropout(p=0.3)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(h2, h3)
        self.drop3 = nn.Dropout(p=0.3)
        self.relu3 = nn.ReLU()

        self.fc4 = nn.Linear(h3, h4)
        self.drop4 = nn.Dropout(p=0.3)
        self.relu4 = nn.ReLU()

        self.out = nn.Linear(h4, n_classes)

        self.freeze_layers(layers_to_freeze)

    def freeze_layers(self, num_layers_to_freeze: int) -> None:
        import re

        freeze_until = min(num_layers_to_freeze, self.num_layers_model)
        print(f"Modelo com {self.num_layers_model} camadas | freezing embeddings + first {freeze_until} encoder layers...")
        for name, param in self.bert.named_parameters():
            if 'embeddings' in name:
                param.requires_grad = False
                continue

            # Suporta nomes como encoder.layer.0... e roberta.encoder.layer.0...
            match = re.search(r'encoder\.layer\.(\d+)\.', name)
            if match:
                layer_num = int(match.group(1))
                param.requires_grad = layer_num >= freeze_until
            else:
                param.requires_grad = True
        print("Layers freezing complete.")

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = output.pooler_output
        if pooled is None:
            # Fallback para arquiteturas sem pooler explícito
            pooled = output.last_hidden_state[:, 0, :]

        hidden = self.fc1(pooled)
        hidden = self.relu1(hidden)
        hidden = self.drop1(hidden)

        hidden = self.fc2(hidden)
        hidden = self.relu2(hidden)
        hidden = self.drop2(hidden)

        hidden = self.fc3(hidden)
        hidden = self.relu3(hidden)
        hidden = self.drop3(hidden)

        hidden = self.fc4(hidden)
        hidden = self.relu4(hidden)
        hidden = self.drop4(hidden)

        return self.out(hidden)


In [ ]:
%%time
import os, gc, copy
from collections import defaultdict
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc

# ---------------------------------------------------------------------------
# Funções auxiliares (parametrizadas)
# ---------------------------------------------------------------------------
def resolve_effective_max_len(tokenizer, model) -> int:
    candidates = []

    tokenizer_limit = getattr(tokenizer, 'model_max_length', None)
    if isinstance(tokenizer_limit, int) and 0 < tokenizer_limit < 100000:
        candidates.append(tokenizer_limit)

    # Suporta tanto backbone HF (tem .config) quanto wrapper customizado (tem .bert.config)
    model_config = getattr(model, 'config', None)
    if model_config is None and hasattr(model, 'bert'):
        model_config = getattr(model.bert, 'config', None)

    model_limit = getattr(model_config, 'max_position_embeddings', None)
    if isinstance(model_limit, int) and model_limit > 0:
        candidates.append(model_limit)

    if not candidates:
        raise ValueError("Nao foi possivel inferir max_len dinamicamente para este modelo/tokenizer.")

    return min(candidates)


def log_stage(message: str) -> None:
    print(f"[ETAPA] {message}")


def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples, scaler, use_amp):
    model.train()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []
    optimizer.zero_grad(set_to_none=True)

    for d in data_loader:
        input_ids = d["input_ids"].to(device, non_blocking=True)
        attention_mask = d["attention_mask"].to(device, non_blocking=True)
        targets = d["targets"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs, targets)

        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def eval_model(model, data_loader, loss_fn, device, n_examples, use_amp=False):
    model.eval()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs, targets)

            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def get_predictions(model, data_loader, device):
    model.eval()
    predictions, prediction_probs, real_values = [], [], []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds)
            prediction_probs.extend(outputs)
            real_values.extend(targets)
    return (torch.stack(predictions).cpu(),
            torch.stack(prediction_probs).cpu(),
            torch.stack(real_values).cpu())


# ---------------------------------------------------------------------------
# Loop principal: fine-tuning + avalia��ão para cada modelo
# ---------------------------------------------------------------------------
all_roc_data = []       # (description, fpr, tpr, roc_auc)
all_metrics = []        # dict por modelo
trained_models = 0

model_dir = f'{BASE_PATH_GENERAL}models'
os.makedirs(model_dir, exist_ok=True)

for idx, model_cfg in enumerate(LISTA_MODEL_CHECKPOINT):
    MODEL_CHECKPOINT = model_cfg['endpoint']
    MODEL_DESC = model_cfg['description']
    LAYERS_TO_FREEZE = model_cfg['layers_to_freeze']
    RUN_FINE_TUNNING = model_cfg.get('fine_tunning', False)
    HEAD_TAIL_APPROACH = model_cfg.get('head_tail_approach', None)

    print("\n" + "=" * 80)
    log_stage(f"Iniciando modelo {idx+1}/{len(LISTA_MODEL_CHECKPOINT)}: {MODEL_DESC}")

    # Verificação de fine_tunning ANTES de qualquer validação de coluna
    if not RUN_FINE_TUNNING:
        print(f"[{idx+1}/{len(LISTA_MODEL_CHECKPOINT)}] SKIP {MODEL_DESC} (fine_tunning=False)")
        continue

    if HEAD_TAIL_APPROACH is None:
        input_text_column = 'RELATORIO'
    elif HEAD_TAIL_APPROACH in (512, '512'):
        input_text_column = 'RELATORIO_HEAD_TAIL_512'
    elif str(HEAD_TAIL_APPROACH).lower() == 'long':
        input_text_column = 'RELATORIO_HEAD_TAIL_LONG'
    else:
        raise ValueError(
            f"head_tail_approach invalido para {MODEL_DESC}: {HEAD_TAIL_APPROACH}. "
            "Valores aceitos: None, 512, 'Long'."
        )

    print(f"[{idx+1}/{len(LISTA_MODEL_CHECKPOINT)}] input_text_column={input_text_column} | head_tail_approach={HEAD_TAIL_APPROACH}")

    if input_text_column not in df_train.columns or input_text_column not in df_test.columns:
        raise KeyError(f"Coluna {input_text_column} nao encontrada em treino e/ou teste.")


    trained_models += 1

    # --- Tokenizer ---
    log_stage(f"Carregando tokenizer: {MODEL_CHECKPOINT}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

    # --- Modelo ---
    log_stage("Instanciando classificador")
    model = HCAcordaoClassifier(len(class_names), LAYERS_TO_FREEZE, MODEL_CHECKPOINT)
    model = model.to(device)

    if not hasattr(model, 'bert') or not hasattr(model.bert, 'config'):
        raise AttributeError(
            f"Modelo {MODEL_DESC} sem backbone/config valido. Esperado: model.bert.config"
        )
    log_stage("Backbone validado com sucesso (model.bert.config)")

    # Inferência dinâmica do limite de tokens por modelo/tokenizer
    log_stage("Inferindo max_len dinamico")
    effective_max_len = resolve_effective_max_len(tokenizer, model.bert)
    print(f"  max_len dinamico: {effective_max_len}")

    # --- DataLoaders ---
    log_stage("Criando DataLoaders")
    train_data_loader = create_data_loader(
        df_train, tokenizer, effective_max_len, BATCH_SIZE,
        shuffle=True, text_column=input_text_column
    )
    test_data_loader = create_data_loader(
        df_test, tokenizer, effective_max_len, BATCH_SIZE,
        shuffle=False, text_column=input_text_column
    )


    # --- Optimizer / Scheduler / Loss ---
    log_stage("Configurando optimizer, scheduler e loss")
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_data_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    loss_fn = nn.CrossEntropyLoss().to(device)
    use_amp = (device.type == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # --- Treinamento ---
    log_stage("Iniciando treinamento")
    history = defaultdict(list)
    best_recall, best_model_weights = 0, None
    epochs_no_improve = 0

    for epoch in range(EPOCHS):
        print(f"\n  Epoch {epoch+1}/{EPOCHS}")
        print("  " + "-" * 60)

        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_data_loader, loss_fn, optimizer, device, scheduler, len(df_train), scaler, use_amp)
        print(f"  Train | loss {train_loss:.4f} acc {train_acc:.4f} prec {train_prec:.4f} rec {train_rec:.4f} f1 {train_f1:.4f}")

        test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
            model, test_data_loader, loss_fn, device, len(df_test), use_amp)
        print(f"  Test  | loss {test_loss:.4f} acc {test_acc:.4f} prec {test_prec:.4f} rec {test_rec:.4f} f1 {test_f1:.4f}")

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)

        if test_rec > best_recall:
            best_model_weights = copy.deepcopy(model.state_dict())
            best_recall = test_rec
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING:
                print(f"  Early Stopping na época {epoch+1}.")
                break

    # --- Gráfico de Loss ---
    plt.figure()
    plt.plot(history['train_loss'], label='train loss')
    plt.plot(history['test_loss'], label='val loss')
    plt.title(f'Perda — {MODEL_DESC}')
    plt.ylabel('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.ylim([0, 2])
    plt.show()

    # --- Carregar melhores pesos ---
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        model_save_path = f'{model_dir}/best_model_{MODEL_CHECKPOINT.replace("/","_")}.bin'
        torch.save(best_model_weights, model_save_path)
        print(f"  Melhor modelo salvo em: {model_save_path}")

    # --- Avaliação Final ---
    log_stage("Executando avaliacao final")
    test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
        model, test_data_loader, loss_fn, device, len(df_test), use_amp)
    print(f"\n  Métricas finais ({MODEL_DESC}):")
    print(f"  Accuracy: {test_acc:.4f} | Precision: {test_prec:.4f} | Recall: {test_rec:.4f} | F1: {test_f1:.4f}")

    # --- Predictions ---
    log_stage("Gerando predicoes no conjunto de teste")
    y_pred, y_pred_probs, y_test = get_predictions(model, test_data_loader, device)

    # --- Relatorio de Classificacao ---
    print(f"\n  Relatorio de Classificacao — {MODEL_DESC}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred)
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    plt.figure()
    hmap = sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    hmap.yaxis.set_ticklabels(hmap.yaxis.get_ticklabels(), rotation=0, ha='right')
    hmap.xaxis.set_ticklabels(hmap.xaxis.get_ticklabels(), rotation=30, ha='right')
    plt.ylabel('Resultado do Julgamento'); plt.xlabel('Decisão Predita')
    plt.title(f'Matriz de Confusão — {MODEL_DESC}')
    plt.show()

    # --- Curva ROC individual ---
    log_stage("Calculando curva ROC individual")
    probs = F.softmax(y_pred_probs, dim=1).numpy()
    y_scores = probs[:, 1]
    fpr, tpr, _ = roc_curve(y_test.numpy(), y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--')
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'Curva ROC — {MODEL_DESC}')
    plt.legend(loc="lower right"); plt.grid(alpha=0.3, linestyle='--')
    plt.show()
    print(f"  AUC: {roc_auc:.4f}")

    # --- Armazena para comparação final ---
    all_roc_data.append((MODEL_DESC, fpr, tpr, roc_auc))
    all_metrics.append({
        'Model': MODEL_DESC,
        'Accuracy': round(test_acc, 4),
        'Precision': round(test_prec, 4),
        'Recall': round(test_rec, 4),
        'F1-Score': round(test_f1, 4),
        'AUC': round(roc_auc, 4),
    })

    # --- Liberar memória ---
    log_stage("Liberando memoria do modelo atual")
    del model, optimizer, scheduler, scaler, tokenizer
    del train_data_loader, test_data_loader
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    log_stage(f"Modelo concluido: {MODEL_DESC}")

print("\n\n" + "=" * 80)
if trained_models == 0:
    print("Nenhum modelo foi treinado (todos estao com fine_tunning=False).")
else:
    print(f"Trade-off concluido para {trained_models} modelo(s).")
print("=" * 80)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: rufimelo/Legal-BERTimbau-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Freezing the embeddings and the first 24 encoder layers of the model...
Layers freezing complete. You can now proceed with training.
1024


## Comparação Final — Curva ROC (Todos os Modelos)

In [ ]:
# --- Curva ROC Comparativa ---
plt.figure(figsize=(10, 8))
colors = plt.cm.get_cmap('tab10', len(all_roc_data))

for i, (desc, fpr, tpr, roc_auc) in enumerate(all_roc_data):
    plt.plot(fpr, tpr, color=colors(i), lw=2, label=f'{desc} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC Comparativa — Trade-off de Modelos', fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## Tabela Comparativa de Métricas

In [ ]:
# --- Tabela de Métricas Finais ---
df_metrics = pd.DataFrame(all_metrics)
df_metrics = df_metrics.sort_values('AUC', ascending=False).reset_index(drop=True)
print("=" * 80)
print("TABELA COMPARATIVA DE MÉTRICAS — TRADE-OFF DE MODELOS")
print("=" * 80)
display(df_metrics.style.highlight_max(
    subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
    color='lightgreen'
))

